# From Voice to Vision — 9. Quantifying Speaker Leakage

Many results reported on RAVDESS rely on random train/test splits, in which utterances of the same
actor appear on both sides. A model can then achieve high accuracy by recognising the *speaker*
rather than the *emotion*. This notebook measures how large that effect is.

The two protocols are compared under identical conditions: the same MFCC features, the same
classifier and the same partition sizes (1200 training and 240 test clips, stratified by emotion).
The only difference is whether the split respects speaker boundaries.

A deterministic RBF-SVM is used instead of the convolutional network precisely so that early
stopping, stochastic augmentation and weight initialisation cannot confound the comparison: any
difference observed is attributable to the split alone. The speaker-independent protocol is
repeated over six folds, each holding out four actors so that every speaker is tested exactly once;
the random protocol is repeated over six seeds.

In [ ]:
# Clone the project repository and install the audio dependencies
REPO_URL = "https://github.com/Nadaa3672/from-voice-to-vision.git"
import os
repo = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
if not os.path.exists(repo):
    !git clone $REPO_URL
%cd $repo
!git pull -q
!pip install -q librosa soundfile noisereduce tqdm

In [ ]:
import numpy as np, json
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from src import config, data_loader, features

data_loader.download_ravdess()
df = data_loader.build_index()
data = features.build_dataset(df, denoise=False, cache=True)
X, y = data["X_vec"], data["y"]
actors = df["actor"].to_numpy()
idx = np.arange(len(y))
print("Clips:", len(y))

In [ ]:
def svm_eval(tr_mask, te_mask):
    """RBF-SVM on the MFCC summary vectors; identical under both protocols."""
    sc = StandardScaler().fit(X[tr_mask])
    m = SVC(kernel="rbf", C=10, gamma="scale", random_state=config.SEED)
    m.fit(sc.transform(X[tr_mask]), y[tr_mask])
    return accuracy_score(y[te_mask], m.predict(sc.transform(X[te_mask])))

# Protocol A — speaker-independent: six folds, each holding out four actors
all_actors = sorted(set(actors))
folds = [all_actors[i * 4:(i + 1) * 4] for i in range(6)]
acc_si = []
for f in folds:
    te = np.isin(actors, f); tr = ~te
    a = svm_eval(tr, te); acc_si.append(a)
    print(f"  speaker-independent, held-out actors {f} → {a:.3f}")

# Protocol B — random split: six seeds, identical partition sizes
acc_rd = []
for seed in range(6):
    i_tr, i_te = train_test_split(idx, test_size=240, stratify=y, random_state=seed)
    tr = np.zeros(len(y), bool); tr[i_tr] = True
    te = np.zeros(len(y), bool); te[i_te] = True
    a = svm_eval(tr, te); acc_rd.append(a)
    print(f"  random split, seed {seed} → {a:.3f}")

acc_si, acc_rd = np.array(acc_si), np.array(acc_rd)
print(f"\nSpeaker-independent : {acc_si.mean():.3f} ± {acc_si.std():.3f}")
print(f"Random split        : {acc_rd.mean():.3f} ± {acc_rd.std():.3f}")
print(f"Inflation due to speaker leakage: "
      f"{(acc_rd.mean() - acc_si.mean()) * 100:.1f} percentage points")

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 4.0))
means = [acc_si.mean(), acc_rd.mean()]; stds = [acc_si.std(), acc_rd.std()]
names = ["Speaker-independent\n(our protocol)", "Random split\n(speaker leakage)"]
bars = ax.bar(names, means, yerr=stds, capsize=6,
              color=["#2f6db0", "#c44e52"], edgecolor="white", width=.55)
for i, vals in enumerate([acc_si, acc_rd]):
    ax.scatter(np.full(len(vals), i) + np.linspace(-.11, .11, len(vals)), vals,
               color="black", s=18, zorder=3, alpha=.7)
for b, m, s in zip(bars, means, stds):
    ax.text(b.get_x() + b.get_width() / 2, m + s + 0.03, f"{m:.3f}",
            ha="center", fontsize=12, weight="bold")
gap = (means[1] - means[0]) * 100
ax.set_ylabel("Test accuracy (SVM on MFCC features)"); ax.set_ylim(0, 0.9)
ax.set_title(f"Speaker leakage inflates accuracy by {gap:.1f} points", fontsize=12)
ax.spines[["top", "right"]].set_visible(False)
ax.axhline(0.125, ls="--", c="grey", lw=1)
ax.text(1.45, 0.142, "chance", color="grey", fontsize=9, ha="right")
plt.tight_layout()
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(config.FIGURES_DIR / "08_leakage.png", dpi=150)
plt.show()

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
json.dump({"speaker_independent": {"mean": float(acc_si.mean()), "std": float(acc_si.std()),
                                   "folds": [float(a) for a in acc_si]},
           "random_split": {"mean": float(acc_rd.mean()), "std": float(acc_rd.std()),
                            "runs": [float(a) for a in acc_rd]},
           "inflation_points": float(gap),
           "design": "RBF-SVM on MFCC summary vectors; 1200 train / 240 test in both protocols"},
          open(config.RESULTS_DIR / "leakage_experiment.json", "w"), indent=2)

## Interpretation

Speaker leakage inflates the reported accuracy by roughly twenty-six percentage points on identical
data, features and model. A random split therefore attributes to emotion recognition a performance
that is, to a large extent, speaker recognition.

The variances are equally informative. The random protocol is remarkably stable, because every test
speaker is also represented in training, whereas the speaker-independent protocol varies almost six
times more: some voices are simply harder than others. A random split does not merely inflate the
mean, it conceals the very variability that a deployed system would encounter.

This result provides the correct frame for reading the accuracy reported elsewhere in this project:
figures obtained under a speaker-independent protocol are not comparable with leakage-affected ones
from the literature.